In [1]:
import numpy as np

In [34]:
def create_training_data():
    """Create the training dataset for tennis prediction."""
    data = [
        ['Sunny', 'Hot', 'High', 'Weak', 'No'],
        ['Sunny', 'Hot', 'High', 'Strong', 'No'],
        ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
        ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
        ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
        ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
        ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
        ['Overcast', 'Mild', 'High', 'Weak', 'No'],
        ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
        ['Rain', 'Mild', 'Normal', 'Weak', 'Yes']
    ]
    return np.array(data)

train_data = create_training_data()
print(train_data)

[['Sunny' 'Hot' 'High' 'Weak' 'No']
 ['Sunny' 'Hot' 'High' 'Strong' 'No']
 ['Overcast' 'Hot' 'High' 'Weak' 'Yes']
 ['Rain' 'Mild' 'High' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Strong' 'No']
 ['Overcast' 'Cool' 'Normal' 'Strong' 'Yes']
 ['Overcast' 'Mild' 'High' 'Weak' 'No']
 ['Sunny' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Mild' 'Normal' 'Weak' 'Yes']]


In [35]:
def compute_prior_probabilities(train_data):
    """
    Calculate prior probabilities P(Play Tennis = Yes/No).

    Args:
        train_data: Training dataset

    Returns:
        Array of prior probabilities [P(No), P(Yes)]
    """
    class_names = ['No', 'Yes']
    total_samples = len(train_data)
    prior_probs = np.zeros(len(class_names))

    ### Your code here
    for i, class_name in enumerate(class_names):
        class_count = np.sum(train_data[:, -1] == class_name)
        prior_probs[i] = class_count / total_samples
    
    return prior_probs

prior_probability = compute_prior_probabilities(train_data)
print("P(“Play Tennis” = No)", prior_probability[0])
print("P(“Play Tennis” = Yes)", prior_probability[1])

P(“Play Tennis” = No) 0.4
P(“Play Tennis” = Yes) 0.6


In [36]:
def compute_conditional_probabilities(train_data):
    """
    Calculate conditional probabilities P(Feature|Class) for all features.

    Args:
        train_data: Training dataset

    Returns:
        Tuple of (conditional_probabilities, feature_values)
    """
    class_names = ['No', 'Yes']
    n_features = train_data.shape[1] - 1  # Exclude target column
    conditional_probs = []
    feature_values = []

    for feature_idx in range(n_features):
        # Get unique values for this feature
        unique_values = np.unique(train_data[:, feature_idx])
        feature_values.append(unique_values)

        # Initialize conditional probability matrix
        feature_cond_probs = np.zeros((len(class_names), len(unique_values)))

        for class_idx, class_name in enumerate(class_names):
            # Get samples for this class
            ### Your code here
            mask = train_data[:, -1] == class_name
            sample = train_data[mask]
            sample_size = len(sample)

            for value_idx, value in enumerate(unique_values):
                # Count occurrences of this feature value in this class
                # Calculate conditional probability
                ### Your code here
                feature_count = np.sum(sample[:, feature_idx] == value) / sample_size
                feature_cond_probs[class_idx][value_idx] = feature_count
                

        conditional_probs.append(feature_cond_probs)

    return conditional_probs, feature_values

In [37]:
_, feature_values  = compute_conditional_probabilities(train_data)
print("x1 = ",feature_values[0])
print("x2 = ",feature_values[1])
print("x3 = ",feature_values[2])
print("x4 = ",feature_values[3])

x1 =  ['Overcast' 'Rain' 'Sunny']
x2 =  ['Cool' 'Hot' 'Mild']
x3 =  ['High' 'Normal']
x4 =  ['Strong' 'Weak']


In [38]:
def get_feature_index(feature_value, feature_values):
    """
    Get the index of a feature value in the feature values array.
    
    Args:
        feature_value: Value to find
        feature_values: Array of possible feature values
        
    Returns:
        Index of the feature value
    """
    return np.where(feature_values == feature_value)[0][0]

_, feature_values = compute_conditional_probabilities(train_data)
outlook = feature_values[0]
i1 = get_feature_index("Overcast", outlook)
i2 = get_feature_index("Rain", outlook)
i3 = get_feature_index("Sunny", outlook)

print(i1, i2, i3)

0 1 2


In [39]:
def train_naive_bayes(train_data):
    """
    Train the Naive Bayes classifier.

    Args:
        train_data: Training dataset

    Returns:
        Tuple of (prior_probabilities, conditional_probabilities, feature_values)
    """

    # Calculate prior probabilities
    prior_probabilities = compute_prior_probabilities(train_data)

    # Calculate conditional probabilities
    conditional_probabilities, feature_values = compute_conditional_probabilities(train_data)

    return prior_probabilities, conditional_probabilities, feature_values

In [40]:
# Train the model
prior_probs, conditional_probs, feature_values = train_naive_bayes(train_data)

In [41]:
# Compute P("Outlook"="Sunny"|Play Tennis"="Yes")
x1 = get_feature_index("Sunny",feature_values[0])
print("P('Outlook'='Sunny'|Play Tennis'='Yes') = ",
      np.round(conditional_probs[0][1, x1],2)
)

P('Outlook'='Sunny'|Play Tennis'='Yes') =  0.17


In [42]:
# Compute P("Outlook"="Sunny"|Play Tennis"="No")
x1 = get_feature_index("Sunny",feature_values[0])
print(
    "P('Outlook'='Sunny'|Play Tennis'='No') = ",
    np.round(conditional_probs[0][0, x1],2)
)

P('Outlook'='Sunny'|Play Tennis'='No') =  0.5


In [43]:
def predict_tennis(
        X, prior_probabilities, conditional_probabilities, feature_values
    ):
    """
    Make a prediction for given features.

    Args:
        X: List of feature values [Outlook, Temperature, Humidity, Wind]
        prior_probabilities: Prior probabilities for each class
        conditional_probabilities: Conditional probabilities for each feature
        feature_values: Names/values for each feature

    Returns:
        Tuple of (prediction, probabilities)
    """
    class_names = ['No', 'Yes']

    # Get feature indices
    feature_indices = []
    for i, feature_value in enumerate(X):
        feature_indices.append(get_feature_index(feature_value, feature_values[i]))

    # Calculate probabilities for each class
    class_probabilities = []

    for class_idx in range(len(class_names)):
        # Start with prior probability
        # Multiply by conditional probabilities
        ### Your code here
        prob = prior_probabilities[class_idx]
        for feature_idx, value_idx in enumerate(feature_indices):
            prob *= conditional_probabilities[feature_idx][class_idx, value_idx]
        class_probabilities.append(prob)
        
    # Normalize probabilities
    total_prob = sum(class_probabilities)
    if total_prob > 0:
        normalized_probs = [p / total_prob for p in class_probabilities]
    else:
        normalized_probs = [0.5, 0.5]  # Default if all probabilities are 0

    # Make prediction
    predicted_class_idx = np.argmax(class_probabilities)
    prediction = class_names[predicted_class_idx]

    # Create probability dictionary
    prob_dict = {
        'No': round(normalized_probs[0].item(), 2),
        'Yes': round(normalized_probs[1].item(), 2)
    }

    return prediction, prob_dict

In [44]:
X = ['Sunny','Cool', 'High', 'Strong']
prior_probs, conditional_probs, feature_values = train_naive_bayes(train_data)
prediction, prob_dict = predict_tennis(
    X, prior_probs, conditional_probs, feature_values
)
if prediction == 'Yes':
    print("AD should go!")
else:
    print("AD should not go!")
prediction, prob_dict

AD should not go!


('No', {'No': 0.87, 'Yes': 0.13})